In [1]:
import csv
import json
import re
import collections
import pandas as pd

In [2]:
SENTENCE_CSV = '/kaggle/input/datasets/arinazamyshevskaya/rlc-raw/annotator_sentence.csv'
ANNOTATION_CSV = '/kaggle/input/datasets/arinazamyshevskaya/rlc-raw/annotator_annotation.csv'
DOCUMENT_CSV = '/kaggle/input/datasets/arinazamyshevskaya/rlc-raw/annotator_document.csv'


In [3]:
def fix_unicode(s):
    # Restore missing-backslash unicode escapes: u0442 -> т
    return re.sub(r'(?<!\\)u([0-9a-fA-F]{4})', lambda m: chr(int(m.group(1), 16)), s)

sentences = {}
with open(SENTENCE_CSV, encoding='utf-8', errors='replace') as f:
    reader = csv.reader(f, escapechar='\\')
    for row in reader:
        if len(row) < 4:
            continue
        sid, text, doc_id, sent_idx = row[0], row[1], row[2], row[3]
        try:
            idx = int(sent_idx)
        except (ValueError, TypeError):
            idx = -1
        sentences[sid] = {'text': text.strip(), 'doc_id': doc_id.strip(), 'sent_idx': idx}

print(f'Loaded {len(sentences)} sentences')


Loaded 198907 sentences


In [4]:
ann_by_sent = collections.defaultdict(list)
with open(ANNOTATION_CSV, encoding='utf-8') as f:
    reader = csv.reader(f, escapechar='\\')
    for row in reader:
        if len(row) < 10:
            continue
        try:
            data = json.loads(row[6])
        except (json.JSONDecodeError, IndexError):
            continue
        sid = row[2]
        corrs = fix_unicode(data.get('corrs', ''))
        quote = fix_unicode(data.get('quote', '')).strip()
        ann_by_sent[sid].append({'quote': quote, 'corrs': corrs})

print(f'Annotations loaded for {len(ann_by_sent)} sentences')

Annotations loaded for 58907 sentences


In [5]:
def reconstruct(text, anns):
    corrected = text
    for ann in anns:
        q, c = ann['quote'], ann['corrs']
        if not q:
            continue  # insertion — skip (no reliable anchor)
        if q in corrected:
            corrected = corrected.replace(q, c, 1)
        elif q.lower() in corrected.lower():
            idx = corrected.lower().find(q.lower())
            corrected = corrected[:idx] + c + corrected[idx + len(q):]
    return re.sub(r'  +', ' ', corrected).strip()

records = []
for sid, anns in ann_by_sent.items():
    s = sentences.get(sid)
    if s is None:
        continue
    text = s['text']
    corrected = reconstruct(text, anns)
    if corrected and corrected != text:
        records.append({
            'sentence_id': sid,
            'document_id': s['doc_id'],
            'sentence_index': s['sent_idx'],
            'text': text,
            'corrected': corrected,
        })

df = pd.DataFrame(records)
print(f'Reconstructed pairs: {len(df)}')
df[['text', 'corrected']].head(5)


Reconstructed pairs: 51776


,text,corrected
0,Студенты так же борятся за спасение озера Байк...,Студенты также борются за спасение озера Байка...
1,Частные институты действуют по средствам рыноч...,Частные институты действуют посредством рыночн...
2,"(? ). (? ).Мы видим на рынке процесс, по средс...","(? ). (? ).Мы видим на рынке процесс, посредст..."
3,У государства экономические и административные...,Государство принимает экономические и админист...
4,Основные задачи государственного регулирования...,Основными задачами государственного регулирова...


In [6]:
# Build lookup: (doc_id, sent_idx) -> plain text for context retrieval
ctx_lookup = {}
for sid, s in sentences.items():
    key = (s['doc_id'], s['sent_idx'])
    ctx_lookup[key] = s['text']

def prev_next(doc_id, sent_idx):
    prev = ctx_lookup.get((doc_id, sent_idx - 1), '')
    nxt  = ctx_lookup.get((doc_id, sent_idx + 1), '')
    return prev, nxt

df[['prev_sent', 'next_sent']] = df.apply(
    lambda r: pd.Series(prev_next(r['document_id'], r['sentence_index'])), axis=1
)

ctx_fill = (df['prev_sent'].str.len() > 0) | (df['next_sent'].str.len() > 0)
print(f'Rows with at least one context sentence: {ctx_fill.sum()} / {len(df)}')


Rows with at least one context sentence: 51725 / 51776


In [7]:
# Filter: keep only Cyrillic-containing sentences
cyrillic = re.compile(r'[а-яА-ЯёЁ]')
df = df[df['text'].str.contains(cyrillic, na=False)]
df = df[df['corrected'].str.contains(cyrillic, na=False)]
df = df.drop_duplicates(subset=['text', 'corrected'])
df = df.reset_index(drop=True)
df.to_csv('sentences_full.csv', index=False)
